In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone
from finhub_api_key import api_key
from datetime import datetime
from term_image.image import from_url
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import numpy as np


In [ ]:
# /stock/market-status?exchange=US


companies = ['AAPL', 'XOM', 'TSLA', 'WMT', 'PFE', 'NFLX', 'JPM', 'CAT']


def get_news(ticker, dfrom, to):
    # /company-news

    params = {
        'token': api_key,
        'symbol': ticker,
        'from': dfrom,
        'to': to
    }

    response = requests.get(
        url='https://finnhub.io/api/v1/company-news', params=params)

    return response.json()


def analyze_news(ticker):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    news = get_news(ticker, now, now)

    prompts = []

    for new in news:
        timestamp = new['datetime']
        date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat().replace('+00:00', 'Z') # перевод в формат iso 

        related = new['related']
        source = new['source']
        summary = new['summary'] if len(new['summary']) > 10 else new['headline']

        model_input = {
            'Benzinga': 0,
            'CNBC': 0,
            'ChartMill': 0,
            'DowJones': 0,
            'MarketWatch': 0,
            'SeekingAlpha': 0,
            'The Guardian': 0,
            'Yahoo': 0,
            'AAPL': 0,
            'TSLA': 0,
            'CAT': 0,
            'JPM': 0,
            'NFLX': 0,
            'PFE': 0,
            'XOM': 0,
            'WMT': 0,
            'webPublicationDate': 0,
            'company': 0,
            'emb_s_0': 0
        }

        ticker_to_name = {
            'AAPL': 'Apple',
            'XOM': 'Exxon',
            'TSLA': 'Tesla',
            'WMT': 'Walmart',
            'PFE': 'Pfizer',
            'NFLX': 'Netflix',
            'JPM': 'JPMorgan',
            'CAT': 'Caterpillar'
        }

        model_input['webPublicationDate'] = date
        model_input[related] = 1
        model_input[source] = 1
        model_input['company'] = ticker_to_name[ticker]
        model_input['emb_s_0'] = summary

        prompts.append(', '.join(str(v) for v in model_input.values()))

    # Инференс промптов через нашу модель

    return prompts


def get_ticker(inp):

    params = {
        'token': api_key,
        'q': inp,
        'exchange': 'US'
    }

    response = requests.get(
        url='https://finnhub.io/api/v1/search', params=params)

    return response.json()['result'][0]['symbol']


def display_image(url):

    image = from_url(url)

    return image


def get_info_by_ticker(ticker):
    #/stock/profile2
    params = {
        'token': api_key,
        'symbol': ticker
    }

    response = requests.get(
        url='https://finnhub.io/api/v1/stock/profile2', params=params)

    return response.json()['logo']

def exchange():
    #/stock/market-status?exchange=US
    params = {
        'token': api_key,
    }

    response = requests.get(
        url='https://finnhub.io/api/v1/stock/market-status?exchange=US', params=params)

    isopen = response.json()["isOpen"]

    if isopen == True: 
        return 'ОТКРЫТА, можно торговать'
    else: return 'ЗАКРЫТА, торговля невозможна'

def get_analytics(ticker):
    # /stock/recommendation?symbol=AAPL

    params = {
        'token': api_key,
        'symbol': ticker
    }

    response = requests.get(
        url='https://finnhub.io/api/v1/stock/recommendation', params=params)

    data = response.json()

    # [{'buy': 23, 'hold': 15, 'period': '2026-04-01', 'sell': 2, 'strongBuy': 14, 'strongSell': 0, 'symbol': 'AAPL'}, {'buy': 22, 'hold': 16, 'period': '2026-03-01', 'sell': 2, 'strongBuy': 14, 'strongSell': 0, 'symbol': 'AAPL'}, {'buy': 21, 'hold': 17, 'period': '2026-02-01', 'sell': 2, 'strongBuy': 14, 'strongSell': 0, 'symbol': 'AAPL'}, {'buy': 21, 'hold': 16, 'period': '2026-01-01', 'sell': 2, 'strongBuy': 14, 'strongSell': 0, 'symbol': 'AAPL'}]

    rec = data[0]

    total = rec['buy'] + rec['hold'] + rec['sell'] + rec['strongBuy'] + rec['strongSell']

    buys = round((rec['buy'] + rec['strongBuy']) / total * 100,1)
    sells = round((rec['sell'] + rec['strongSell']) / total * 100 ,1)

    all = [buys,sells, total]

    return all

    


In [ ]:
name = 'apple'

ticker = get_ticker(name)
print(f'\nНовостной анализ для ${ticker}\n')

logo = get_info_by_ticker(ticker=ticker)


img = np.array(Image.open(urllib.request.urlopen(logo)))
plt.imshow(img)
plt.axis('off')
plt.show()

news = analyze_news(ticker)

recs = get_analytics(ticker)

an_rec = 'НЕОПРЕДЕЛЕННОСТЬ'

if recs[0] - 20 > recs[1]: 
    an_rec = 'ПОКУПАТЬ'

elif recs[1] - 20 > recs[0]:
    an_rec = 'ПРОДАВАТЬ'


print(f'По оценкам аналитиков: \n \n{recs[0]}% рекомендуют ПОКУПАТЬ \n {recs[1]}% рекомендуют ПРОДАВАТЬ \n Консенус аналитиков: {an_rec}')
print(f'Рынок сейчас -- {exchange()}')
print(news)

In [ ]:
df = pd.read_csv('data_s.csv')
df.head(5)